# 04 - 替换模型算子：OOT 插件机制

> **本 Notebook 涵盖内容**
> - Llama 模型中算子的调用链分析
> - OOT (Out-of-Tree) 替换机制详解
> - 用 Triton 实现替换 Llama 的 SiluAndMul
> - 用 Triton 实现替换 Llama 的 RMSNorm
> - PluggableLayer 替换机制
> - 端到端集成验证

**运行示例**：将前面实现的 Triton kernel 注册为 OOT 算子，替换 Llama 模型中的原有实现。

## 1. Llama 模型的算子调用链

让我们先看看 Llama 模型中使用了哪些算子，以及它们是如何被调用的：

![Llama 模型调用链](../diagrams/04-llama-call-chain.svg)

**关键替换点**（标红/标蓝的部分）：
1. **SiluAndMul**（红色）：MLP 层的激活函数，每个 Decoder Layer 调用 1 次
2. **RMSNorm**（蓝色）：每个 Decoder Layer 调用 2 次 + 最后 1 次 = 2N+1 次

对于 Llama-7B（N=32 层），每次推理调用：
- SiluAndMul: 32 次
- RMSNorm: 65 次

> **Source**: vllm/model_executor/models/llama.py:81-334

## 2. OOT 替换机制详解

vLLM 的 OOT (Out-of-Tree) 机制允许你在不修改 vLLM 源码的情况下替换任何已注册的 CustomOp。

![OOT 替换机制](../diagrams/04-oot-replacement.svg)

关键代码：

```python
# vllm/model_executor/custom_op.py:109-128

class CustomOp(nn.Module):
    def __new__(cls, *args, **kwargs):
        op_name = cls.__name__

        # 核心逻辑：检查是否有 OOT 替换
        if op_name not in op_registry_oot:
            op_cls_to_instantiate = cls
        else:
            op_cls_to_instantiate = op_registry_oot[op_name]
            logger.debug("Using OOT: %s -> %s", op_name, op_cls_to_instantiate)

        return super().__new__(op_cls_to_instantiate)
```

这个机制的精妙之处：**调用方代码完全不需要修改**。当 Llama 模型中写的是 `self.act_fn = SiluAndMul()` 时，如果你注册了 OOT 替换，实际创建的是你的自定义类。

> **Source**: vllm/model_executor/custom_op.py:109-128

## 3. 实战：构建 OOT 替换系统

我们先构建一个模拟 vLLM 注册系统的完整框架，然后替换 Llama 的算子。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import triton
import triton.language as tl
from typing import Optional, Tuple

device = torch.device("cuda")

# ==== 模拟 vLLM 的注册系统 ====
op_registry = {}
op_registry_oot = {}


class CustomOp(nn.Module):
    """模拟 vLLM 的 CustomOp 基类"""

    def __new__(cls, *args, **kwargs):
        op_name = cls.__name__
        if op_name in op_registry_oot:
            replacement = op_registry_oot[op_name]
            print(f"  [OOT] {op_name} -> {replacement.__name__}")
            return super().__new__(replacement)
        return super().__new__(cls)

    def __init__(self, **kwargs):
        super().__init__()
        if torch.cuda.is_available():
            self._forward_method = self.forward_cuda
        else:
            self._forward_method = self.forward_native

    def forward(self, *args, **kwargs):
        return self._forward_method(*args, **kwargs)

    def forward_native(self, *args, **kwargs):
        raise NotImplementedError

    def forward_cuda(self, *args, **kwargs):
        raise NotImplementedError

    @classmethod
    def register(cls, name):
        def decorator(op_cls):
            op_cls.name = name
            op_registry[name] = op_cls
            return op_cls
        return decorator

    @classmethod
    def register_oot(cls, _cls=None, name=None):
        def decorator(op_cls):
            reg_name = name or cls.__name__
            op_registry_oot[reg_name] = op_cls
            return op_cls
        if _cls is None:
            return decorator
        return decorator(_cls)

print("CustomOp framework initialized")

## 4. 定义「原始」算子（模拟 vLLM 内置的实现）

In [ ]:
# ==== 原始算子：使用 PyTorch 实现 ====

@CustomOp.register("silu_and_mul")
class SiluAndMul(CustomOp):
    """vLLM 原始的 SiluAndMul（简化版）"""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def forward_native(self, x: torch.Tensor) -> torch.Tensor:
        d = x.shape[-1] // 2
        return F.silu(x[..., :d]) * x[..., d:]

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        # 原始实现：调用 CUDA C++ kernel
        # 这里简化为 PyTorch native
        return self.forward_native(x)


@CustomOp.register("rms_norm")
class RMSNorm(CustomOp):
    """vLLM 原始的 RMSNorm（简化版）"""

    def __init__(self, hidden_size: int, eps: float = 1e-6, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.variance_epsilon = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward_native(
        self, x: torch.Tensor, residual: Optional[torch.Tensor] = None
    ) -> torch.Tensor | tuple:
        if residual is not None:
            x = x + residual
            residual = x
        orig_dtype = x.dtype
        x = x.float()
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.variance_epsilon)
        x = (x * self.weight.float()).to(orig_dtype)
        if residual is not None:
            return x, residual
        return x

    def forward_cuda(self, x, residual=None):
        return self.forward_native(x, residual)


print(f"Registered ops: {list(op_registry.keys())}")

# 测试原始算子
x = torch.randn(4, 512, device=device, dtype=torch.float16)
op = SiluAndMul()
print(f"SiluAndMul output shape: {op(x).shape}")

norm = RMSNorm(256).half().to(device)
x_n = torch.randn(4, 256, device=device, dtype=torch.float16)
print(f"RMSNorm output shape: {norm(x_n).shape}")

## 5. 编写 Triton 替换实现

现在，让我们用 Triton 编写高性能替换：

In [ ]:
# ==== Triton Kernels ====

@triton.jit
def triton_silu_and_mul_kernel(
    output_ptr, o_stride,
    input_ptr, i_stride,
    d,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(axis=0)
    col_block = tl.program_id(axis=1)
    offsets = col_block * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < d

    inp = input_ptr + row * i_stride
    out = output_ptr + row * o_stride

    gate = tl.load(inp + offsets, mask=mask).to(tl.float32)
    up = tl.load(inp + offsets + d, mask=mask).to(tl.float32)
    result = gate * tl.sigmoid(gate) * up
    tl.store(out + offsets, result.to(input_ptr.dtype.element_ty), mask=mask)


@triton.jit
def triton_rms_norm_kernel(
    output_ptr, o_stride,
    input_ptr, i_stride,
    weight_ptr,
    hidden_size,
    eps: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(axis=0)
    inp = input_ptr + row * i_stride
    out = output_ptr + row * o_stride

    # 计算 RMS
    sum_sq = tl.zeros([1], dtype=tl.float32)
    for off in range(0, hidden_size, BLOCK_SIZE):
        offsets = off + tl.arange(0, BLOCK_SIZE)
        mask = offsets < hidden_size
        x = tl.load(inp + offsets, mask=mask).to(tl.float32)
        sum_sq += tl.sum(x * x, axis=0)

    rms = tl.rsqrt(sum_sq / hidden_size + eps)

    # 归一化
    for off in range(0, hidden_size, BLOCK_SIZE):
        offsets = off + tl.arange(0, BLOCK_SIZE)
        mask = offsets < hidden_size
        x = tl.load(inp + offsets, mask=mask).to(tl.float32)
        w = tl.load(weight_ptr + offsets, mask=mask).to(tl.float32)
        result = x * rms * w
        tl.store(out + offsets, result.to(input_ptr.dtype.element_ty), mask=mask)


print("Triton kernels defined")

## 6. 注册 OOT 替换

In [ ]:
# ==== OOT 替换：用 Triton 替换原有实现 ====

@CustomOp.register_oot(name="SiluAndMul")
class TritonSiluAndMul(SiluAndMul):
    """
    使用 Triton 的 SiluAndMul 替换实现

    继承自原始 SiluAndMul，只覆盖 forward_cuda
    """

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2:
            orig_shape = x.shape
            x = x.view(-1, x.shape[-1])
        else:
            orig_shape = None

        b, n = x.shape
        d = n // 2
        output = torch.empty((b, d), dtype=x.dtype, device=x.device)
        BLOCK_SIZE = 1024
        grid = (b, triton.cdiv(d, BLOCK_SIZE))
        triton_silu_and_mul_kernel[grid](
            output, output.stride(0),
            x, x.stride(0),
            d, BLOCK_SIZE=BLOCK_SIZE,
        )
        if orig_shape is not None:
            return output.view(orig_shape[:-1] + (d,))
        return output


@CustomOp.register_oot(name="RMSNorm")
class TritonRMSNorm(RMSNorm):
    """
    使用 Triton 的 RMSNorm 替换实现

    继承自原始 RMSNorm，只覆盖 forward_cuda
    """

    def forward_cuda(
        self, x: torch.Tensor, residual: Optional[torch.Tensor] = None
    ):
        if residual is not None:
            # 对于 fused add+rmsnorm，回退到 native（Triton 版本在 notebook 03）
            return self.forward_native(x, residual)

        if x.ndim != 2:
            orig_shape = x.shape
            x = x.view(-1, x.shape[-1])
        else:
            orig_shape = None

        b = x.shape[0]
        hidden_size = x.shape[-1]
        output = torch.empty_like(x)
        BLOCK_SIZE = min(1024, triton.next_power_of_2(hidden_size))

        triton_rms_norm_kernel[(b,)](
            output, output.stride(0),
            x, x.stride(0),
            self.weight,
            hidden_size,
            eps=self.variance_epsilon,
            BLOCK_SIZE=BLOCK_SIZE,
        )
        if orig_shape is not None:
            return output.view(orig_shape)
        return output


print(f"OOT registry: {list(op_registry_oot.keys())}")
print("\nNow when you create SiluAndMul(), it uses TritonSiluAndMul!")
print("When you create RMSNorm(), it uses TritonRMSNorm!")

## 7. 验证 OOT 替换生效

In [ ]:
# 当你调用 SiluAndMul() 时，OOT 机制自动替换为 TritonSiluAndMul
print("Creating SiluAndMul():")
act = SiluAndMul()
print(f"  Actual class: {act.__class__.__name__}")
print(f"  Is Triton version: {isinstance(act, TritonSiluAndMul)}")

print("\nCreating RMSNorm(256):")
norm = RMSNorm(256).half().to(device)
print(f"  Actual class: {norm.__class__.__name__}")
print(f"  Is Triton version: {isinstance(norm, TritonRMSNorm)}")

# 验证正确性
print("\n--- SiluAndMul Verification ---")
x = torch.randn(128, 8192, device=device, dtype=torch.float16)
result = act(x)
expected = F.silu(x[:, :4096]) * x[:, 4096:]
max_err = (result - expected).abs().max().item()
print(f"Max error: {max_err:.2e}")
assert max_err < 1e-2
print("PASSED!")

print("\n--- RMSNorm Verification ---")
x = torch.randn(128, 256, device=device, dtype=torch.float16)
result = norm(x)
# Reference
x_f = x.float()
variance = x_f.pow(2).mean(dim=-1, keepdim=True)
expected = (x_f * torch.rsqrt(variance + 1e-6) * norm.weight.float()).half()
max_err = (result - expected).abs().max().item()
print(f"Max error: {max_err:.2e}")
assert max_err < 1e-2
print("PASSED!")

## 8. 端到端：替换后的 Llama 模型

In [ ]:
# ==== 简化版 Llama 模型 ====

class LlamaMLP(nn.Module):
    """
    对应 vllm/model_executor/models/llama.py:81-121
    """
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_up_proj = nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = SiluAndMul()  # OOT 自动替换为 TritonSiluAndMul!

    def forward(self, x):
        x = self.gate_up_proj(x)
        x = self.act_fn(x)
        x = self.down_proj(x)
        return x


class LlamaDecoderLayer(nn.Module):
    """
    对应 vllm/model_executor/models/llama.py:253-333
    """
    def __init__(self, hidden_size, intermediate_size, num_heads=8):
        super().__init__()
        self.hidden_size = hidden_size
        self.input_layernorm = RMSNorm(hidden_size)  # OOT 自动替换!
        self.post_attention_layernorm = RMSNorm(hidden_size)  # OOT 自动替换!
        self.mlp = LlamaMLP(hidden_size, intermediate_size)

        # 简化的 attention（用线性层代替）
        self.self_attn = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, hidden_states, residual=None):
        # Self Attention
        if residual is None:
            residual = hidden_states
            hidden_states = self.input_layernorm(hidden_states)
        else:
            hidden_states, residual = self.input_layernorm(hidden_states, residual)
        hidden_states = self.self_attn(hidden_states)

        # MLP
        hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


print("Building Llama model with OOT-replaced operators...")
print("=" * 60)

hidden_size = 256
intermediate_size = 512
num_layers = 4

layers = nn.ModuleList([
    LlamaDecoderLayer(hidden_size, intermediate_size)
    for _ in range(num_layers)
])
final_norm = RMSNorm(hidden_size)

# Move to GPU
layers = layers.half().to(device)
final_norm = final_norm.half().to(device)

print("=" * 60)

# 打印算子使用情况
for i, layer in enumerate(layers):
    print(f"\nLayer {i}:")
    print(f"  input_layernorm: {layer.input_layernorm.__class__.__name__}")
    print(f"  post_attn_norm:  {layer.post_attention_layernorm.__class__.__name__}")
    print(f"  act_fn:          {layer.mlp.act_fn.__class__.__name__}")

print(f"\nFinal norm: {final_norm.__class__.__name__}")

In [ ]:
# 端到端推理测试
x = torch.randn(4, 16, hidden_size, device=device, dtype=torch.float16)
residual = None
hidden_states = x

print("Running forward pass through 4-layer Llama...")
for i, layer in enumerate(layers):
    hidden_states, residual = layer(hidden_states, residual)
    print(f"  Layer {i}: hidden_states shape = {hidden_states.shape}")

# Final norm
output = final_norm(hidden_states + residual if residual is not None else hidden_states)
print(f"\nFinal output shape: {output.shape}")
print(f"Output dtype: {output.dtype}")
print(f"Has NaN: {torch.isnan(output).any().item()}")
print(f"Has Inf: {torch.isinf(output).any().item()}")

# 梯度检查
loss = output.sum()
loss.backward()
print(f"\nBackward pass completed!")
print(f"Layer 0 MLP gate_up_proj grad: {layers[0].mlp.gate_up_proj.weight.grad is not None}")
print("\nAll checks PASSED!")

## 9. 性能对比：OOT Triton vs 原始实现

In [ ]:
# 清除 OOT 注册，测量原始实现
op_registry_oot_backup = op_registry_oot.copy()
op_registry_oot.clear()

# 创建使用原始实现的算子
original_silu = SiluAndMul()
original_norm = RMSNorm(4096).half().to(device)
print(f"Original SiluAndMul: {original_silu.__class__.__name__}")
print(f"Original RMSNorm: {original_norm.__class__.__name__}")

# 恢复 OOT 注册
op_registry_oot.update(op_registry_oot_backup)

# 创建使用 Triton 的算子
triton_silu = SiluAndMul()
triton_norm = RMSNorm(4096).half().to(device)
print(f"\nTriton SiluAndMul: {triton_silu.__class__.__name__}")
print(f"Triton RMSNorm: {triton_norm.__class__.__name__}")

# 性能对比
x_silu = torch.randn(128, 8192, device=device, dtype=torch.float16)
x_norm = torch.randn(128, 4096, device=device, dtype=torch.float16)

t_orig_silu = triton.testing.do_bench(lambda: original_silu(x_silu))
t_triton_silu = triton.testing.do_bench(lambda: triton_silu(x_silu))
t_orig_norm = triton.testing.do_bench(lambda: original_norm(x_norm))
t_triton_norm = triton.testing.do_bench(lambda: triton_norm(x_norm))

print(f"\n{'='*50}")
print(f"Performance Comparison:")
print(f"{'='*50}")
print(f"SiluAndMul:")
print(f"  Original:  {t_orig_silu:.4f} ms")
print(f"  Triton:    {t_triton_silu:.4f} ms")
print(f"  Speedup:   {t_orig_silu/t_triton_silu:.2f}x")
print(f"\nRMSNorm:")
print(f"  Original:  {t_orig_norm:.4f} ms")
print(f"  Triton:    {t_triton_norm:.4f} ms")
print(f"  Speedup:   {t_orig_norm/t_triton_norm:.2f}x")

## 10. 实际的 vLLM OOT 插件开发

在真实的 vLLM 环境中，OOT 插件通常以 Python 包的形式发布：

```
my-vllm-plugin/
+-- setup.py
+-- my_vllm_plugin/
    +-- __init__.py       # 在这里注册 OOT 算子
    +-- custom_ops.py     # Triton kernel 实现
    +-- platform.py       # 可选：自定义平台检测
```

```python
# my_vllm_plugin/__init__.py
from vllm.model_executor.layers.activation import SiluAndMul
from vllm.model_executor.custom_op import CustomOp

@CustomOp.register_oot(name="SiluAndMul")
class MyTritonSiluAndMul(SiluAndMul):
    def forward_cuda(self, x):
        # 你的 Triton 实现
        ...
```

安装后，vLLM 会自动发现并加载你的插件。

**真实案例**：vLLM 的测试中就有一个 OOT 插件示例：

```python
# tests/plugins/vllm_add_dummy_platform/vllm_add_dummy_platform/dummy_custom_ops.py

from vllm.model_executor.layers.rotary_embedding import RotaryEmbedding

@RotaryEmbedding.register_oot
class DummyRotaryEmbedding(RotaryEmbedding):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.addition_config = True  # 添加自定义配置

    def forward_oot(self, *args, **kwargs) -> tuple:
        return super().forward_oot(*args, **kwargs)
```

> **Source**: tests/plugins/vllm_add_dummy_platform/vllm_add_dummy_platform/dummy_custom_ops.py

## 11. PluggableLayer: 另一种替换机制

除了 CustomOp，vLLM 还有 PluggableLayer 用于更复杂的层级替换：

```python
# vllm/model_executor/custom_op.py:32-100

class PluggableLayer(nn.Module):
    """
    模块组合抽象：可以包含子模块，有自己的状态。
    与 CustomOp 的区别：不提供 forward_cuda/forward_native 分发，
    而是整体替换整个层的类。
    """

    def __new__(cls, *args, **kwargs):
        layer_name = cls.__name__
        if layer_name in op_registry_oot:
            return super().__new__(op_registry_oot[layer_name])
        return super().__new__(cls)
```

| 特性 | CustomOp | PluggableLayer |
|------|----------|---------------|
| 分发方式 | 按平台分发 forward_* | 整体替换类 |
| 状态 | 通常无状态 | 可以有子模块和参数 |
| 替换粒度 | 单个操作 | 整个层（含子模块） |
| 典型用途 | 激活函数、归一化 | 注意力层、MLP 层 |

> **Source**: vllm/model_executor/custom_op.py:32-100

## 12. 本章小结

| 概念 | 说明 |
|------|------|
| OOT (Out-of-Tree) | 不修改 vLLM 源码的外部扩展机制 |
| `register_oot` | 注册替换实现到 `op_registry_oot` |
| `__new__` 拦截 | 在实例化时检查 OOT 注册表，透明替换 |
| 继承覆盖 | OOT 类继承原始类，只覆盖需要改的方法 |
| PluggableLayer | 整个层级的替换机制 |
| 插件包 | 以 Python 包形式分发 OOT 算子 |

## 源码映射表

| 本教程实现 | vLLM 原始源码 | 章节 |
|-----------|-------------|------|
| `CustomOp.__new__` OOT | `custom_op.py:109-128` | 2, 3 |
| `register_oot` | `custom_op.py:332-353` | 2, 6 |
| `TritonSiluAndMul` | `activation.py:117-148 (SiluAndMul)` | 6 |
| `TritonRMSNorm` | `layernorm.py:107-356 (RMSNorm)` | 6 |
| `LlamaMLP` | `models/llama.py:81-121` | 8 |
| `LlamaDecoderLayer` | `models/llama.py:253-333` | 8 |
| 真实 OOT 插件 | `tests/plugins/.../dummy_custom_ops.py` | 10 |

## 下一步

在 [05-full-integration.ipynb](05-full-integration.ipynb) 中，我们将把所有知识整合在一起，完成一个完整的端到端示例。